# Week 4 Exercise - Multi-Language Test Case Generator

Write or paste source code, select a language and LLM model, then generate and run unit tests.

**Supported languages:** Python, JavaScript, TypeScript, Java, Ruby, Go, Rust, C++, C, C#, PHP, Swift, Kotlin

In [ ]:
# imports

import os
import re
import subprocess
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# environment and client setup

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

openai_client = OpenAI()
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODELS = {
    "gpt-oss": {"client": ollama_client, "model": "gpt-oss"},
    "llama3.2": {"client": ollama_client, "model": "llama3.2"},
    "gpt-5-nano": {"client": openai_client, "model": "gpt-5-nano"},
}

RUNNERS = {
    "python": (".py", lambda f: ["python", f]),
    "javascript": (".js", lambda f: ["node", f]),
    "typescript": (".ts", lambda f: ["npx", "ts-node", f]),
    "ruby": (".rb", lambda f: ["ruby", f]),
    "php": (".php", lambda f: ["php", f]),
    "swift": (".swift", lambda f: ["swift", f]),
}

EDITOR_LANGUAGES = [
    "python", "javascript", "typescript", "java", "ruby", "go",
    "rust", "cpp", "c", "csharp", "php", "swift", "kotlin",
]

In [ ]:
# core logic

def build_system_prompt(language):
    return (
        f"You are an expert {language} developer specializing in unit testing. "
        f"Given source code, generate a maximum of 7 unit tests covering normal behavior, "
        f"edge cases, and error handling. "
        f"Output ONLY valid, runnable {language} code with no markdown fences and no prose outside of code comments. "
        f"Include the original source code so the test file is self-contained and executable. "
        f"Do not modify, rewrite, or refactor the original source code; write tests against it as-is. "
        f"Configure the test runner for verbose output (e.g., unittest.main(verbosity=2) in Python). "
        f"Do NOT guess behavior. If behavior is ambiguous, skip that test. Generate only high-confidence tests with deterministic assertions. "
        f"Name tests with clear function-style identifiers (prefer test_* naming) so results can display exact test names."
    )


def clean_llm_output(text):
    """Strip markdown code fences that LLMs sometimes add despite instructions."""
    text = text.strip()
    if text.startswith("```"):
        first_nl = text.find("\n")
        if first_nl != -1:
            text = text[first_nl + 1:]
    if text.endswith("```"):
        text = text[:-3].rstrip()
    return text


def generate_tests(code, model_name, language):
    config = MODELS[model_name]
    system_prompt = build_system_prompt(language)
    user_prompt = (
        f"Generate comprehensive unit tests for this {language} code. "
        f"Treat the source as read-only and do not edit the original implementation.\n\n{code}"
    )

    request_args = {
        "model": config["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": True,
    }

    # OpenAI models require max_completion_tokens; Ollama typically expects max_tokens.
    if model_name == "llama3.2":
        request_args["max_tokens"] = 4096
    else:
        request_args["max_completion_tokens"] = 4096

    stream = config["client"].chat.completions.create(**request_args)

    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield clean_llm_output(reply)


def run_tests(test_code, language):
    runner = RUNNERS.get(language)
    if not runner:
        return f"Automatic test execution is not supported for '{language}'. Copy the generated tests and run them manually."

    suffix, cmd_fn = runner
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=suffix, delete=False)
    try:
        tmp.write(test_code)
        tmp.close()
        result = subprocess.run(
            cmd_fn(tmp.name), capture_output=True, text=True, timeout=30
        )
        output = (result.stdout + "\n" + result.stderr).strip()
        return output or "Tests completed with no output."
    except subprocess.TimeoutExpired:
        return "Test execution timed out (30s limit)."
    except FileNotFoundError as e:
        return f"Runtime not found: {e}\nMake sure the {language} runtime is installed."
    finally:
        os.unlink(tmp.name)


def extract_test_name(raw_name):
    name = raw_name.strip()

    # pytest node id: path::Class::test_function
    if "::" in name:
        tail = name.split("::")[-1].strip()
        if tail:
            return tail

    # unittest verbose often has: test_xxx (ClassName)
    if "(" in name:
        head = name.split("(", 1)[0].strip()
        if re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", head):
            return head

    # Prefer explicit function-like identifiers when present.
    m = re.search(r"\b(test[_A-Za-z0-9]+)\b", name)
    if m:
        return m.group(1)

    # If no function-style identifier exists, keep full label (do not truncate).
    return name


def format_test_results(raw_output):
    lines = raw_output.splitlines()
    parsed = []

    for line in lines:
        text = line.strip()

        # unittest verbose format: test_name (Class) ... ok|FAIL|ERROR
        m = re.match(r"^(.+?)\s+\.{3}\s+(ok|FAIL|ERROR)$", text)
        if m:
            test_name = extract_test_name(m.group(1).strip())
            status = m.group(2)
            parsed.append((test_name, status == "ok"))
            continue

        # pytest format: path::class::test_name PASSED|FAILED|ERROR
        m = re.match(r"^(.+?)\s+(PASSED|FAILED|ERROR)$", text)
        if m:
            test_name = extract_test_name(m.group(1).strip())
            status = m.group(2)
            parsed.append((test_name, status == "PASSED"))
            continue

        # generic check/cross output
        m = re.match(r"^[✓✔]\s+(.+)$", text)
        if m:
            parsed.append((extract_test_name(m.group(1).strip()), True))
            continue
        m = re.match(r"^[✗✘×]\s+(.+)$", text)
        if m:
            parsed.append((extract_test_name(m.group(1).strip()), False))
            continue

    if not parsed:
        return raw_output

    passed = sum(1 for _, ok in parsed if ok)
    failed = len(parsed) - passed

    ran_line = ""
    for line in lines:
        m = re.search(r"Ran\s+(\d+)\s+tests?\s+in\s+([0-9.]+s)", line)
        if m:
            ran_line = f"Ran {m.group(1)} tests in {m.group(2)}"
            break
    if not ran_line:
        ran_line = f"Ran {len(parsed)} tests"

    result_lines = []
    for idx, (name, ok) in enumerate(parsed, start=1):
        icon = "✅" if ok else "❌"
        result_lines.append(f"{idx}. {icon} {name}")

    if failed == 0:
        final_status = "All tests passed"
    else:
        final_status = f"{passed} passed and {failed} failed"

    result_lines.extend([
        "",
        "Final note:",
        f"• {ran_line}",
        f"• {final_status}",
    ])

    return "\n".join(result_lines)

In [ ]:
# gradio UI


def on_editor_lang_change(language, current_code):
    return language, gr.Code(value=current_code, language=language)


def on_generate(code, model_name, lang_state):
    if not code or not code.strip():
        yield gr.Code(value="No source code provided. Type or paste code in the editor.", language="python")
        return
    if not lang_state:
        yield gr.Code(value="Language not selected. Choose a language from the dropdown.", language="python")
        return

    for chunk in generate_tests(code, model_name, lang_state):
        yield gr.Code(value=chunk, language=lang_state)


def on_run_tests(test_code, lang_state):
    if not test_code or not test_code.strip():
        return "No test code to run. Generate tests first."
    if not lang_state:
        return "Language not selected. Choose a language from the dropdown."

    raw = run_tests(test_code, lang_state)
    return format_test_results(raw)


CUSTOM_CSS = """
#generated-tests-box {
  max-height: 80vh;
}
#generated-tests-box .cm-editor,
#generated-tests-box .cm-scroller,
#generated-tests-box textarea {
  max-height: 80vh !important;
  overflow: auto !important;
}
"""

with gr.Blocks(title="Test Case Generator", theme=gr.themes.Soft(), css=CUSTOM_CSS) as ui:
    gr.Markdown("# Test Case Generator\nWrite code, select a language, generate tests with an LLM, and run them.")
    lang_state = gr.State(value="python")

    with gr.Row(equal_height=True):
        # Left column — code editor + language selector + model selector + generate button
        with gr.Column(scale=1):
            code_display = gr.Code(
                label="Source Code",
                language="python",
                lines=18,
                interactive=True,
            )
            editor_lang = gr.Dropdown(
                choices=EDITOR_LANGUAGES,
                value="python",
                label="Language",
            )
            model_dropdown = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="gpt-oss",
                label="LLM Model",
            )
            generate_btn = gr.Button(
                "Validate & Generate Tests", variant="primary", size="lg"
            )

        # Middle column — generated tests (85%), run button (15%)
        with gr.Column(scale=1):
            test_output = gr.Code(
                label="Generated Test Cases",
                language="python",
                lines=22,
                interactive=True,
                elem_id="generated-tests-box",
            )
            run_btn = gr.Button("Run Tests", variant="secondary", size="lg")

        # Right column — test execution results
        with gr.Column(scale=1):
            test_results = gr.Textbox(
                label="Test Results",
                lines=25,
                max_lines=40,
                interactive=False,
                show_copy_button=True,
            )

    editor_lang.change(
        fn=on_editor_lang_change,
        inputs=[editor_lang, code_display],
        outputs=[lang_state, code_display],
    )
    generate_btn.click(
        on_generate,
        inputs=[code_display, model_dropdown, lang_state],
        outputs=[test_output],
    )
    run_btn.click(
        on_run_tests, inputs=[test_output, lang_state], outputs=[test_results],
    )

ui.launch(inbrowser=True)